In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor

In [3]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


In [4]:
check = train[
    ["battle_turn", "round", "turn", "previous_hp", "pikachu_hp"]
].copy()

check["next_previous_hp"] = check["previous_hp"].shift(-1)

check["hp_matches"] = (
    check["pikachu_hp"] == check["next_previous_hp"]
)

print(check.head(20))
print("\nMatch percentage:",
      check["hp_matches"].mean() * 100)

             battle_turn  round  turn  previous_hp  pikachu_hp  \
0   Round 1 - Turn 00:00      1     0         54.0          54   
1   Round 1 - Turn 01:00      1     1         31.0          11   
2   Round 1 - Turn 02:00      1     2          7.0          12   
3   Round 1 - Turn 03:00      1     3         54.0           0   
4   Round 1 - Turn 04:00      1     4          NaN          29   
5   Round 1 - Turn 05:00      1     5         34.0          15   
6   Round 1 - Turn 06:00      1     6         54.0          54   
7   Round 1 - Turn 07:00      1     7         54.0          47   
8   Round 1 - Turn 08:00      1     8         18.0          31   
9   Round 1 - Turn 09:00      1     9         54.0          54   
10  Round 1 - Turn 10:00      1    10         54.0          54   
11  Round 1 - Turn 11:00      1    11         54.0          11   
12  Round 1 - Turn 12:00      1    12         17.0           2   
13  Round 1 - Turn 13:00      1    13         12.0           0   
14  Round 

In [5]:
print(train[[
    "battle_turn",
    "round",
    "turn",
    "opponent_pokemon",
    "previous_hp",
    "pikachu_hp"
]].head(30).to_string(index=False))

         battle_turn  round  turn opponent_pokemon  previous_hp  pikachu_hp
Round 1 - Turn 00:00      1     0        Charizard         54.0          54
Round 1 - Turn 01:00      1     1        Dragonite         31.0          11
Round 1 - Turn 02:00      1     2          Steelix          7.0          12
Round 1 - Turn 03:00      1     3        Charizard         54.0           0
Round 1 - Turn 04:00      1     4         Alakazam          NaN          29
Round 1 - Turn 05:00      1     5        Charizard         34.0          15
Round 1 - Turn 06:00      1     6         Venusaur         54.0          54
Round 1 - Turn 07:00      1     7        Blastoise         54.0          47
Round 1 - Turn 08:00      1     8         Alakazam         18.0          31
Round 1 - Turn 09:00      1     9        Dragonite         54.0          54
Round 1 - Turn 10:00      1    10         Vaporeon         54.0          54
Round 1 - Turn 11:00      1    11          Machamp         54.0          11
Round 1 - Tu

In [6]:
print(
    train.groupby("round")["turn"]
    .agg(["min", "max", "count"])
    .head(20)
)

       min  max  count
round                 
1        0   23     24
2        0   23     24
3        0   23     24
4        0   23     24
5        0   23     24
6        0   23     24
7        0   23     24
8        0   23     24
9        0   23     24
10       0   23     24
11       0   23     24
12       0   23     24
13       0   23     24
14       0   23     24
15       0   23     24
16       0   23     24
17       0   23     24
18       0   23     24
19       0   23     24
20       0   23     24


In [77]:
print(
    train[train["round"]==1][
        ["round","turn","opponent_level","pikachu_level","move_power","attack_stat","defense_stat","sp_attack_stat","sp_defense_stat","speed_stat_pikachu","speed_stat_opponent","attack_stage","defense_stage","speed_stage",
        "opponent_type","max_hp","previous_hp", "pikachu_hp"]
    ].to_string(index=False)
)

 round  turn  opponent_level  pikachu_level  move_power  attack_stat  defense_stat  sp_attack_stat  sp_defense_stat  speed_stat_pikachu  speed_stat_opponent  attack_stage  defense_stage  speed_stage opponent_type  max_hp  previous_hp  pikachu_hp
     1     0            27.0           26.0        90.0          NaN          37.0            29.0             43.0                48.0                 29.0           0.0            0.0          0.0          Fire    54.0         54.0          54
     1     1            32.0           26.0        80.0         34.0          26.0            39.0             49.0                55.0                 25.0           0.0           -1.0          0.0        Dragon    54.0         31.0          11
     1     2            24.0           26.0        40.0         30.0          31.0            33.0             38.0                46.0                 47.0          -1.0            1.0          0.0         Steel    54.0          7.0          12
     1     3    

In [60]:
print(train["held_item"].unique())

<StringArray>
['Sitrus Berry', <NA>, 'Assault Vest', 'Light Ball', 'Focus Sash']
Length: 5, dtype: string


In [72]:
print(train["pikachu_ability"].unique())

<StringArray>
['Static', 'Lightning Rod', <NA>]
Length: 3, dtype: string


In [68]:
print(train["weather_condition"].unique())

<StringArray>
['Clear', 'Rain', 'Sandstorm', 'Electric Terrain', 'Sun', <NA>, 'Hail']
Length: 7, dtype: string


In [ ]:
X = train.drop(columns=["pikachu_hp", "battle_turn"])
X_test = test.drop(columns=["battle_turn"])
target = "pikachu_hp"

# Clean column names
train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

# Clean categorical text
categorical_cols = train.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    if col in test.columns:
        train[col] = train[col].astype("string").str.strip()
        test[col] = test[col].astype("string").str.strip()

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_11924\754274685.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train.select_dtypes(include=["object"]).columns


In [22]:
for df in [train, test]:

    # Damage relative to Pikachu's maximum HP
    df["damage_pct"] = (
        df["damage_dealt"] /
        df["max_hp"].replace(0, np.nan)
    )

    # Healing relative to max HP
    df["healing_pct"] = (
        df["healing_applied"] /
        df["max_hp"].replace(0, np.nan)
    )

    # Previous HP relative to maximum HP
    df["previous_hp_pct"] = (
        df["previous_hp"] /
        df["max_hp"].replace(0, np.nan)
    )

In [23]:
for df in [train, test]:

    df["move_vs_opponent"] = (
        df["move_type"].fillna("Unknown")
        + "_vs_"
        + df["opponent_type"].fillna("Unknown")
    )

In [24]:
for df in [train, test]:

    df["effective_move_power"] = (
        df["move_power"] *
        df["type_effectiveness"]
    )

In [31]:
for df in [train, test]:

    df["effective_damage_potential"] = (
        df["effective_move_power"] *
        df["move_hit"]
    )

In [32]:
for df in [train, test]:

    df["turn_squared"] = df["turn"] ** 2

    df["turn_sqrt"] = np.sqrt(df["turn"])

In [33]:
train = train.sort_values(
    ["round", "turn"]
).reset_index(drop=True)

test = test.sort_values(
    ["round", "turn"]
).reset_index(drop=True)

In [34]:
lag_columns = [
    "damage_dealt",
    "healing_applied",
    "previous_hp",
    "type_effectiveness",
    "move_power"
]

for col in lag_columns:

    train[f"{col}_lag1"] = (
        train.groupby("round")[col].shift(1)
    )

    test[f"{col}_lag1"] = (
        test.groupby("round")[col].shift(1)
    )

In [35]:
for col in [
    "damage_dealt",
    "healing_applied",
    "previous_hp"
]:

    train[f"{col}_lag2"] = (
        train.groupby("round")[col].shift(2)
    )

    test[f"{col}_lag2"] = (
        test.groupby("round")[col].shift(2)
    )

In [36]:
train["damage_last_3"] = (
    train.groupby("round")["damage_dealt"]
    .transform(
        lambda x: x.shift(1).rolling(3).sum()
    )
)

test["damage_last_3"] = (
    test.groupby("round")["damage_dealt"]
    .transform(
        lambda x: x.shift(1).rolling(3).sum()
    )
)

In [37]:
for df in [train, test]:

    df.replace(
        [np.inf, -np.inf],
        np.nan,
        inplace=True
    )

In [38]:
numeric_features = train.select_dtypes(
    include=["int64", "float64"]
).columns

correlations = (
    train[numeric_features]
    .corr()["pikachu_hp"]
    .sort_values(ascending=False)
)

print(correlations)

pikachu_hp                    1.000000
trainer_focus_score           0.961998
previous_hp                   0.443397
previous_hp_pct               0.419830
damage_dealt_lag1             0.156475
max_hp                        0.131141
pikachu_level                 0.130375
speed_stat_pikachu            0.118681
opponent_level                0.116936
healing_applied               0.111496
damage_last_3                 0.110127
sp_defense_stat               0.107362
healing_pct                   0.106723
defense_stage                 0.101842
defense_stat                  0.100808
sp_attack_stat                0.092669
attack_stat                   0.091297
type_effectiveness_lag1       0.086384
speed_stat_opponent           0.051040
move_power_lag1               0.039503
damage_dealt_lag2             0.024913
experience_points             0.021611
healing_applied_lag1          0.021361
healing_applied_lag2          0.008463
turn_sqrt                     0.007511
turn                     

In [39]:
engineered = [
    "damage_pct",
    "healing_pct",
    "previous_hp_pct",
    "effective_move_power",
    "effective_damage_potential",
    "turn_squared",
    "turn_sqrt",
    "damage_dealt_lag1",
    "damage_dealt_lag2",
    "healing_applied_lag1",
    "previous_hp_lag1",
    "damage_last_3"
]

print(
    train[engineered + ["pikachu_hp"]]
    .corr()["pikachu_hp"]
    .sort_values(ascending=False)
)


pikachu_hp                    1.000000
previous_hp_pct               0.419830
damage_dealt_lag1             0.156475
damage_last_3                 0.110127
healing_pct                   0.106723
damage_dealt_lag2             0.024913
healing_applied_lag1          0.021361
turn_sqrt                     0.007511
turn_squared                  0.006105
previous_hp_lag1              0.005295
effective_move_power         -0.323775
damage_pct                   -0.346297
effective_damage_potential   -0.375413
Name: pikachu_hp, dtype: float64


In [40]:
print(
    train[engineered + ["pikachu_hp"]]
    .corr()["pikachu_hp"]
    .sort_values(ascending=False)
)

pikachu_hp                    1.000000
previous_hp_pct               0.419830
damage_dealt_lag1             0.156475
damage_last_3                 0.110127
healing_pct                   0.106723
damage_dealt_lag2             0.024913
healing_applied_lag1          0.021361
turn_sqrt                     0.007511
turn_squared                  0.006105
previous_hp_lag1              0.005295
effective_move_power         -0.323775
damage_pct                   -0.346297
effective_damage_potential   -0.375413
Name: pikachu_hp, dtype: float64


In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


# ============================================================
# 1. LOAD DATA
# ============================================================

target = "pikachu_hp"


# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

categorical_cols = train.select_dtypes(include=["object"]).columns

for col in categorical_cols:
    train[col] = (
        train[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

    if col in test.columns:
        test[col] = (
            test[col]
            .fillna("Unknown")
            .astype(str)
            .str.strip()
        )

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

for df in [train, test]:

    # Damage relative to maximum HP
    df["damage_pct"] = (
        df["damage_dealt"] /
        df["max_hp"].replace(0, np.nan)
    )

    # Healing relative to maximum HP
    df["healing_pct"] = (
        df["healing_applied"] /
        df["max_hp"].replace(0, np.nan)
    )

    # Previous HP relative to maximum HP
    df["previous_hp_pct"] = (
        df["previous_hp"] /
        df["max_hp"].replace(0, np.nan)
    )

    # Move type × opponent type
    df["move_vs_opponent"] = (
        df["move_type"].fillna("Unknown")
        + "_vs_"
        + df["opponent_type"].fillna("Unknown")
    )

    # Move power adjusted for effectiveness
    df["effective_move_power"] = (
        df["move_power"] *
        df["type_effectiveness"]
    )

    # Move power × effectiveness × whether it hit
    df["effective_damage_potential"] = (
        df["effective_move_power"] *
        df["move_hit"]
    )

    # Time features
    df["turn_squared"] = df["turn"] ** 2
    df["turn_sqrt"] = np.sqrt(df["turn"])


# ============================================================
# 4. SORT BY ROUND AND TURN
# ============================================================

train = train.sort_values(
    ["round", "turn"]
).reset_index(drop=True)

test = test.sort_values(
    ["round", "turn"]
).reset_index(drop=True)


# ============================================================
# 5. LAG FEATURES
# ============================================================

lag_columns = [
    "damage_dealt",
    "healing_applied",
    "previous_hp",
    "type_effectiveness",
    "move_power"
]

for col in lag_columns:

    train[f"{col}_lag1"] = (
        train.groupby("round")[col].shift(1)
    )

    test[f"{col}_lag1"] = (
        test.groupby("round")[col].shift(1)
    )


# Two-turn lag

for col in [
    "damage_dealt",
    "healing_applied",
    "previous_hp"
]:

    train[f"{col}_lag2"] = (
        train.groupby("round")[col].shift(2)
    )

    test[f"{col}_lag2"] = (
        test.groupby("round")[col].shift(2)
    )


# ============================================================
# 6. RECENT DAMAGE
# ============================================================

train["damage_last_3"] = (
    train.groupby("round")["damage_dealt"]
    .transform(
        lambda x: x.shift(1).rolling(3).sum()
    )
)

test["damage_last_3"] = (
    test.groupby("round")["damage_dealt"]
    .transform(
        lambda x: x.shift(1).rolling(3).sum()
    )
)

# ============================================================
# 7. REMOVE INFINITE VALUES
# ============================================================

train.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

test.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)


# ============================================================
# 8. X AND y
# ============================================================

X = train.drop(columns=["pikachu_hp", "battle_turn"])
X_test = test.drop(columns=["battle_turn"])
y = train[target]



# ============================================================
# 9. IDENTIFY FEATURE TYPES
# ============================================================

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Number of numerical features:",
      len(numeric_features))

print("Number of categorical features:",
      len(categorical_features))


# ============================================================
# 10. PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),

    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


# ============================================================
# 11. MODEL
# ============================================================

model = Pipeline([
    ("preprocessor", preprocessor),

    ("regression", LinearRegression())
])


# ============================================================
# 12. TEMPORAL VALIDATION
# ============================================================

split = int(len(X) * 0.90)

X_train = X.iloc[:split]
X_valid = X.iloc[split:]

y_train = y.iloc[:split]
y_valid = y.iloc[split:]

print("\nTraining rows:", len(X_train))
print("Validation rows:", len(X_valid))


# ============================================================
# 13. TRAIN
# ============================================================

model.fit(
    X_train,
    y_train
)


# ============================================================
# 14. VALIDATION
# ============================================================

valid_predictions = model.predict(
    X_valid
)

validation_r2 = r2_score(
    y_valid,
    valid_predictions
)

print("\nDay 3 Validation R²:",
      validation_r2)


# ============================================================
# 15. TRAIN ON FULL DATA
# ============================================================

model.fit(
    X,
    y
)


# ============================================================
# 16. PREDICT TEST DATA
# ============================================================

test_predictions = model.predict(
    X_test
)


# ============================================================
# 17. CREATE SUBMISSION
# ============================================================

sample = pd.read_csv(
    "sample_submission.csv"
)

submission = pd.DataFrame({
    "battle_turn": sample["battle_turn"],
    "pikachu_hp": test_predictions
})


submission.to_csv(
    "submissionDay4.csv",
    index=False
)


# ============================================================
# 18. CHECK SUBMISSION
# ============================================================

print("\nSubmission preview:")
print(submission.head())

print("\nSubmission shape:")
print(submission.shape)

print("\nMissing values:")
print(submission.isnull().sum())

print("\nPrediction statistics:")
print(submission["pikachu_hp"].describe())

Number of numerical features: 40
Number of categorical features: 0

Training rows: 72014
Validation rows: 8002

Day 3 Validation R²: 0.9496718585555229

Submission preview:
               battle_turn  pikachu_hp
0  Round 1828 - Turn 00:00   48.313387
1  Round 1828 - Turn 01:00   48.439877
2  Round 1828 - Turn 02:00   29.861181
3  Round 1828 - Turn 03:00   26.701024
4  Round 1828 - Turn 04:00   18.511976

Submission shape:
(240, 2)

Missing values:
battle_turn    0
pikachu_hp     0
dtype: int64

Prediction statistics:
count    240.000000
mean      38.709392
std       17.756970
min        6.643468
25%       24.418561
50%       39.598180
75%       53.436706
max       69.337757
Name: pikachu_hp, dtype: float64


In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score


# ============================================================
# 1. LOAD DATA
# ============================================================

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

target = "pikachu_hp"

train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()


# ============================================================
# 2. CLEAN CATEGORICAL FEATURES
# ============================================================

categorical_cols = train.select_dtypes(include=["object"]).columns

for col in categorical_cols:

    train[col] = (
        train[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )

    if col in test.columns:
        test[col] = (
            test[col]
            .fillna("Unknown")
            .astype(str)
            .str.strip()
        )


# ============================================================
# 3. FEATURE ENGINEERING FUNCTION
# ============================================================

def create_features(df):

    df = df.copy()

    # --------------------------------------------------------
    # A. BASIC HP / DAMAGE FEATURES
    # --------------------------------------------------------

    df["damage_pct"] = (
        df["damage_dealt"] /
        df["max_hp"].replace(0, np.nan)
    )

    df["healing_pct"] = (
        df["healing_applied"] /
        df["max_hp"].replace(0, np.nan)
    )

    df["previous_hp_pct"] = (
        df["previous_hp"] /
        df["max_hp"].replace(0, np.nan)
    )


    # --------------------------------------------------------
    # B. POKEMON TYPE MATCHUP
    # --------------------------------------------------------

    df["move_vs_opponent"] = (
        df["move_type"].fillna("Unknown")
        + "_vs_"
        + df["opponent_type"].fillna("Unknown")
    )


    # --------------------------------------------------------
    # C. MOVE EFFECTIVENESS
    # --------------------------------------------------------

    df["effective_move_power"] = (
        df["move_power"] *
        df["type_effectiveness"]
    )

    df["effective_damage_potential"] = (
        df["move_power"] *
        df["type_effectiveness"] *
        df["move_hit"]
    )


    # --------------------------------------------------------
    # D. LEVEL RELATIONSHIPS
    # --------------------------------------------------------

    df["level_difference"] = (
        df["pikachu_level"] -
        df["opponent_level"]
    )

    df["level_ratio"] = (
        df["pikachu_level"] /
        df["opponent_level"].replace(0, np.nan)
    )


    # --------------------------------------------------------
    # E. ATTACK / DEFENSE RELATIONSHIPS
    # --------------------------------------------------------

    df["attack_difference"] = (
        df["attack_stat"] -
        df["defense_stat"]
    )

    df["attack_ratio"] = (
        df["attack_stat"] /
        df["defense_stat"].replace(0, np.nan)
    )


    # Special attack vs special defense

    df["sp_attack_difference"] = (
        df["sp_attack_stat"] -
        df["sp_defense_stat"]
    )

    df["sp_attack_ratio"] = (
        df["sp_attack_stat"] /
        df["sp_defense_stat"].replace(0, np.nan)
    )


    # --------------------------------------------------------
    # F. SPEED RELATIONSHIP
    # --------------------------------------------------------

    df["speed_difference"] = (
        df["speed_stat_pikachu"] -
        df["speed_stat_opponent"]
    )

    df["speed_ratio"] = (
        df["speed_stat_pikachu"] /
        df["speed_stat_opponent"].replace(0, np.nan)
    )


    # --------------------------------------------------------
    # G. STAGE-ADJUSTED STATS
    # --------------------------------------------------------

    df["attack_stage_effect"] = (
        df["attack_stat"] *
        df["attack_stage"]
    )

    df["defense_stage_effect"] = (
        df["defense_stat"] *
        df["defense_stage"]
    )

    df["speed_stage_effect"] = (
        df["speed_stat_pikachu"] *
        df["speed_stage"]
    )


    # --------------------------------------------------------
    # H. EFFECTIVE ATTACK / DEFENSE
    # --------------------------------------------------------

    df["effective_attack_vs_defense"] = (
        df["attack_stat"] *
        df["attack_stage"]
    ) / (
        df["defense_stat"] *
        df["defense_stage"]
    ).replace(0, np.nan)


    # --------------------------------------------------------
    # I. MOVE × BATTLE STAT INTERACTIONS
    # --------------------------------------------------------

    df["move_power_x_attack"] = (
        df["move_power"] *
        df["attack_stat"]
    )

    df["move_power_x_level"] = (
        df["move_power"] *
        df["pikachu_level"]
    )

    df["effectiveness_x_power"] = (
        df["type_effectiveness"] *
        df["move_power"]
    )

    df["effectiveness_x_attack"] = (
        df["type_effectiveness"] *
        df["attack_stat"]
    )

    df["effectiveness_x_level"] = (
        df["type_effectiveness"] *
        df["pikachu_level"]
    )


    # --------------------------------------------------------
    # J. TIME FEATURES
    # --------------------------------------------------------

    df["turn_squared"] = df["turn"] ** 2

    df["turn_sqrt"] = np.sqrt(
        df["turn"].clip(lower=0)
    )

    #terrain features
    df["move_category_weather"] = (
    df["move_category"].fillna("Unknown")
    + "_"
    + df["weather_condition"].fillna("Unknown")
    )

    df["move_category_terrain"] = (
        df["move_category"].fillna("Unknown")
        + "_"
        + df["terrain_type"].fillna("Unknown")
    )

    df["status_weather"] = (
        df["pikachu_status"].fillna("Unknown")
        + "_"
        + df["weather_condition"].fillna("Unknown")
    )

    df["ability_weather"] = (
        df["pikachu_ability"].fillna("Unknown")
        + "_"
        + df["weather_condition"].fillna("Unknown")
    )

    df["ability_terrain"] = (
        df["pikachu_ability"].fillna("Unknown")
        + "_"
        + df["terrain_type"].fillna("Unknown")
    )


    return df


# ============================================================
# 4. CREATE THE NON-TEMPORAL FEATURES
# ============================================================

train = create_features(train)
test = create_features(test)


# ============================================================
# 5. SORT BY ROUND + TURN
# ============================================================

train = train.sort_values(
    ["round", "turn"]
).reset_index(drop=True)

test = test.sort_values(
    ["round", "turn"]
).reset_index(drop=True)


# ============================================================
# 6. TEMPORAL LAG FEATURES
# ============================================================

lag_columns = [
    "damage_dealt",
    "healing_applied",
    "previous_hp",
    "type_effectiveness",
    "move_power"
]

for col in lag_columns:

    train[f"{col}_lag1"] = (
        train.groupby("round")[col].shift(1)
    )

    test[f"{col}_lag1"] = (
        test.groupby("round")[col].shift(1)
    )


# ============================================================
# 7. TWO-TURN LAG
# ============================================================

lag2_columns = [
    "damage_dealt",
    "healing_applied",
    "previous_hp"
]

for col in lag2_columns:

    train[f"{col}_lag2"] = (
        train.groupby("round")[col].shift(2)
    )

    test[f"{col}_lag2"] = (
        test.groupby("round")[col].shift(2)
    )


# ============================================================
# 8. ROLLING RECENT DAMAGE
# ============================================================

train["damage_last_3"] = (
    train.groupby("round")["damage_dealt"]
    .transform(
        lambda x:
        x.shift(1).rolling(3).sum()
    )
)

test["damage_last_3"] = (
    test.groupby("round")["damage_dealt"]
    .transform(
        lambda x:
        x.shift(1).rolling(3).sum()
    )
)


# ============================================================
# 9. REMOVE INF / -INF
# ============================================================

train.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

test.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)


# ============================================================
# 10. CREATE X AND y
# ============================================================

X = train.drop(
    columns=[target, "battle_turn"],
    errors="ignore"
)

y = train[target]

X_test = test.drop(
    columns=["battle_turn"],
    errors="ignore"
)


# ============================================================
# 11. FIND NUMERICAL + CATEGORICAL FEATURES
# ============================================================

numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical columns:")
print(categorical_features)


# ============================================================
# 12. PREPROCESSING
# ============================================================

numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),

    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])


preprocessor = ColumnTransformer([

    (
        "num",
        numeric_pipeline,
        numeric_features
    ),

    (
        "cat",
        categorical_pipeline,
        categorical_features
    )
])


# ============================================================
# 13. LINEAR REGRESSION MODEL
# ============================================================

model = Pipeline([

    (
        "preprocessor",
        preprocessor
    ),

    (
        "regression",
        LinearRegression()
    )
])


# ============================================================
# 14. TEMPORAL VALIDATION
# ============================================================

split = int(len(X) * 0.90)

X_train = X.iloc[:split]
X_valid = X.iloc[split:]

y_train = y.iloc[:split]
y_valid = y.iloc[split:]

print("\nTraining rows:", len(X_train))
print("Validation rows:", len(X_valid))


# ============================================================
# 15. TRAIN
# ============================================================

model.fit(
    X_train,
    y_train
)


# ============================================================
# 16. VALIDATION PREDICTIONS
# ============================================================

valid_predictions = model.predict(
    X_valid
)

def temporal_test(train_end, valid_start, valid_end):

    X_train = X.iloc[:train_end]
    y_train = y.iloc[:train_end]

    X_valid = X.iloc[valid_start:valid_end]
    y_valid = y.iloc[valid_start:valid_end]

    # Train
    model.fit(X_train, y_train)

    # Predict
    predictions = model.predict(X_valid)

    # Score
    score = r2_score(y_valid, predictions)

    return score


# ============================================================
# CREATE TEMPORAL WINDOWS
# ============================================================

n = len(X)

print("Total rows:", n)


# 70% train -> next 10% validation
score_70 = temporal_test(
    int(n * 0.70),
    int(n * 0.70),
    int(n * 0.80)
)


# 80% train -> next 10% validation
score_80 = temporal_test(
    int(n * 0.80),
    int(n * 0.80),
    int(n * 0.90)
)


# 90% train -> final 10% validation
score_90 = temporal_test(
    int(n * 0.90),
    int(n * 0.90),
    n
)


# ============================================================
# RESULTS
# ============================================================

print("\n================================")
print("TEMPORAL VALIDATION RESULTS")
print("================================")

print("70% → next 10% R²:", score_70)
print("80% → next 10% R²:", score_80)
print("90% → final 10% R²:", score_90)


# ============================================================
# 17. TRAIN ON ALL DATA
# ============================================================

model.fit(
    X,
    y
)


# ============================================================
# 18. TEST PREDICTIONS
# ============================================================

test_predictions = model.predict(
    X_test
)


# ============================================================
# 19. CREATE SUBMISSION
# ============================================================

submission = pd.DataFrame({

    "battle_turn": sample["battle_turn"],

    "pikachu_hp": test_predictions
})


submission.to_csv(
    "submissionDay4.csv",
    index=False
)


# ============================================================
# 20. CHECK SUBMISSION
# ============================================================

print("\nSubmission:")
print(submission.head())

print("\nShape:")
print(submission.shape)

print("\nMissing values:")
print(submission.isnull().sum())

print("\nPrediction statistics:")
print(
    submission["pikachu_hp"].describe()
)

C:\Users\Bhavin\AppData\Local\Temp\ipykernel_21632\1844273226.py:30: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = train.select_dtypes(include=["object"]).columns
C:\Users\Bhavin\AppData\Local\Temp\ipykernel_21632\1844273226.py:399: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org

Numerical features: 57
Categorical features: 16

Categorical columns:
['opponent_pokemon', 'opponent_type', 'move_used', 'move_type', 'move_category', 'weather_condition', 'pikachu_status', 'terrain_type', 'held_item', 'pikachu_ability', 'move_vs_opponent', 'move_category_weather', 'move_category_terrain', 'status_weather', 'ability_weather', 'ability_terrain']

Training rows: 72014
Validation rows: 8002
Total rows: 80016

TEMPORAL VALIDATION RESULTS
70% → next 10% R²: 0.9472887402845883
80% → next 10% R²: 0.94978686089928
90% → final 10% R²: 0.9498902474903657

Submission:
               battle_turn  pikachu_hp
0  Round 1828 - Turn 00:00   46.149241
1  Round 1828 - Turn 01:00   46.928079
2  Round 1828 - Turn 02:00   30.825140
3  Round 1828 - Turn 03:00   26.931227
4  Round 1828 - Turn 04:00   17.800779

Shape:
(240, 2)

Missing values:
battle_turn    0
pikachu_hp     0
dtype: int64

Prediction statistics:
count    240.000000
mean      38.697298
std       17.676946
min        5.208243


In [2]:
print("TRAIN ROUND RANGE:")
print(train["round"].min(), train["round"].max())

print("\nTEST ROUND RANGE:")
print(test["round"].min(), test["round"].max())

print("\nTRAIN TURN RANGE:")
print(train["turn"].min(), train["turn"].max())

print("\nTEST TURN RANGE:")
print(test["turn"].min(), test["turn"].max())

TRAIN ROUND RANGE:
1 3334

TEST ROUND RANGE:
1828 1837

TRAIN TURN RANGE:
0 23

TEST TURN RANGE:
0 23


In [3]:
print(train[train["round"].between(1818, 1837)][
    ["round", "turn", "previous_hp", "pikachu_hp"]
].to_string(index=False))

 round  turn  previous_hp  pikachu_hp
  1818     0         59.0          44
  1818     1         40.0          36
  1818     2         40.0          12
  1818     3         15.0          15
  1818     4         15.0           0
  1818     5         59.0          16
  1818     6         40.0          31
  1818     7         31.0           0
  1818     8          2.0           0
  1818     9         59.0          15
  1818    10         52.0          30
  1818    11         52.0          16
  1818    12         46.0           0
  1818    13         59.0           6
  1818    14         37.0           8
  1818    15         37.0          18
  1818    16         37.0           0
  1818    17         59.0          15
  1818    18         50.0          25
  1818    19         42.0          12
  1818    20         28.0           0
  1818    21         59.0           0
  1818    22         16.0           0
  1818    23          5.0           0
  1819     0          NaN           3
  1819     1

In [4]:
print("Rows around test boundary:")
print(
    train[train["round"].between(1820, 1827)]
    [["round", "turn", "previous_hp", "pikachu_hp"]]
    .tail(30)
)

Rows around test boundary:
       round  turn  previous_hp  pikachu_hp
43818   1826    18         78.0           0
43819   1826    19         78.0          43
43820   1826    20         35.0          78
43821   1826    21         78.0          56
43822   1826    22         63.0          66
43823   1826    23         54.0          30
43824   1827     0          5.0           6
43825   1827     1         54.0           3
43826   1827     2         25.0           0
43827   1827     3         54.0           0
43828   1827     4          6.0          23
43829   1827     5          6.0           6
43830   1827     6         54.0          33
43831   1827     7         54.0          22
43832   1827     8         12.0           4
43833   1827     9         54.0          22
43834   1827    10         26.0           0
43835   1827    11          5.0          14
43836   1827    12         54.0          33
43837   1827    13         30.0           0
43838   1827    14         54.0          54
43839

In [5]:
print("Last 10 training rounds:")
print(sorted(train["round"].unique())[-10:])

print("\nFirst 10 training rounds:")
print(sorted(train["round"].unique())[:10])

print("\nTest rounds:")
print(sorted(test["round"].unique()))

Last 10 training rounds:
[np.int64(3325), np.int64(3326), np.int64(3327), np.int64(3328), np.int64(3329), np.int64(3330), np.int64(3331), np.int64(3332), np.int64(3333), np.int64(3334)]

First 10 training rounds:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

Test rounds:
[np.int64(1828), np.int64(1829), np.int64(1830), np.int64(1831), np.int64(1832), np.int64(1833), np.int64(1834), np.int64(1835), np.int64(1836), np.int64(1837)]


In [8]:
print(
    train[train["round"]==1828][
        ["round","turn","opponent_level","pikachu_level","move_power","attack_stat","defense_stat","sp_attack_stat","sp_defense_stat","speed_stat_pikachu","speed_stat_opponent","attack_stage","defense_stage","speed_stage",
        "opponent_type","max_hp","previous_hp", "pikachu_hp"]
    ].to_string(index=False)
)


 round  turn  opponent_level  pikachu_level  move_power  attack_stat  defense_stat  sp_attack_stat  sp_defense_stat  speed_stat_pikachu  speed_stat_opponent  attack_stage  defense_stage  speed_stage opponent_type  max_hp  previous_hp  pikachu_hp
  1828     0            42.0           42.0       120.0         54.0          37.0            55.0             46.0                83.0                 56.0          -1.0           -1.0         -2.0         Ghost    81.0         10.0           0
  1828     1            46.0           42.0         0.0         48.0          48.0            48.0             59.0                80.0                 69.0           1.0            2.0          0.0         Grass    81.0         81.0          81
  1828     2            47.0           42.0       120.0         59.0          51.0            45.0             56.0                85.0                 93.0           0.0            0.0         -1.0         Water    81.0         81.0           0
  1828     3    

In [7]:
print(
    test[test["round"]==1828][
        ["round","turn","opponent_level","pikachu_level","move_power","attack_stat","defense_stat","sp_attack_stat","sp_defense_stat","speed_stat_pikachu","speed_stat_opponent","attack_stage","defense_stage","speed_stage",
        "opponent_type","max_hp","previous_hp"]
    ].to_string(index=False)
)

 round  turn  opponent_level  pikachu_level  move_power  attack_stat  defense_stat  sp_attack_stat  sp_defense_stat  speed_stat_pikachu  speed_stat_opponent  attack_stage  defense_stage  speed_stage opponent_type  max_hp  previous_hp
  1828     0            43.0           42.0       120.0         59.0          45.0            52.0             49.0                84.0                 55.0          -1.0            0.0          1.0       Psychic    81.0         78.0
  1828     1            41.0           42.0        90.0         60.0          45.0            48.0             43.0                87.0                 60.0           0.0            0.0          2.0         Ghost    81.0         36.0
  1828     2            44.0           42.0        20.0         48.0          47.0            52.0             58.0                80.0                 41.0          -2.0           -2.0         -2.0          Fire    81.0         81.0
  1828     3            41.0           42.0        20.0         

In [9]:
test_rounds = sorted(test["round"].unique())

print("Test rounds:")
print(test_rounds)

validation_mask = train["round"].isin(test_rounds)

X_train = X[~validation_mask]
y_train = y[~validation_mask]

X_valid = X[validation_mask]
y_valid = y[validation_mask]

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))

model.fit(X_train, y_train)

predictions = model.predict(X_valid)

score = r2_score(y_valid, predictions)

print("\nKaggle-like validation R²:", score)

Test rounds:
[np.int64(1828), np.int64(1829), np.int64(1830), np.int64(1831), np.int64(1832), np.int64(1833), np.int64(1834), np.int64(1835), np.int64(1836), np.int64(1837)]
Training rows: 79776
Validation rows: 240

Kaggle-like validation R²: 0.9447636788686016


In [10]:
print(
    test[
        [
            "round",
            "turn",
            "damage_dealt_lag1",
            "damage_dealt_lag2",
            "previous_hp_lag1",
            "type_effectiveness_lag1",
            "move_power_lag1"
        ]
    ].head(30).to_string(index=False)
)

 round  turn  damage_dealt_lag1  damage_dealt_lag2  previous_hp_lag1  type_effectiveness_lag1  move_power_lag1
  1828     0                NaN                NaN               NaN                      NaN              NaN
  1828     1               75.0                NaN              78.0                      1.0            120.0
  1828     2               55.0               75.0              36.0                      1.0             90.0
  1828     3               10.0               55.0              81.0                      1.0             20.0
  1828     4                NaN               10.0              72.0                      1.0             20.0
  1828     5                0.0                NaN              78.0                      0.5            110.0
  1828     6               40.0                0.0              78.0                      1.0             40.0
  1828     7                5.0               40.0              36.0                      1.0             20.0
 

In [11]:
print("\nMissing lag values in TEST:")

print(
    test[
        [
            "damage_dealt_lag1",
            "damage_dealt_lag2",
            "previous_hp_lag1",
            "type_effectiveness_lag1",
            "move_power_lag1"
        ]
    ].isnull().sum()
)


Missing lag values in TEST:
damage_dealt_lag1          17
damage_dealt_lag2          26
previous_hp_lag1           20
type_effectiveness_lag1    16
move_power_lag1            17
dtype: int64


In [12]:
cols = [
    "round",
    "turn",
    "damage_dealt",
    "damage_dealt_lag1",
    "previous_hp",
    "previous_hp_lag1",
    "type_effectiveness",
    "type_effectiveness_lag1",
    "move_power",
    "move_power_lag1"
]

print(
    test[
        cols
    ][
        test["damage_dealt_lag1"].isna()
        | test["previous_hp_lag1"].isna()
        | test["move_power_lag1"].isna()
    ].to_string(index=False)
)

 round  turn  damage_dealt  damage_dealt_lag1  previous_hp  previous_hp_lag1  type_effectiveness  type_effectiveness_lag1  move_power  move_power_lag1
  1828     0          75.0                NaN         78.0               NaN                 1.0                      NaN       120.0              NaN
  1828     4           0.0                NaN         78.0              72.0                 0.5                      1.0       110.0             20.0
  1828     9          50.0                NaN         81.0              81.0                 NaN                      2.0       110.0             40.0
  1829     0         150.0                NaN         81.0               NaN                 1.0                      NaN       110.0              NaN
  1829    19          60.0               55.0         50.0               NaN                 2.0                      2.0       100.0             40.0
  1830     0          15.0                NaN         81.0               NaN                 2

In [13]:
print(
    test.groupby("round")[
        [
            "damage_dealt_lag1",
            "previous_hp_lag1",
            "move_power_lag1"
        ]
    ].apply(lambda x: x.isna().sum())
)

       damage_dealt_lag1  previous_hp_lag1  move_power_lag1
round                                                      
1828                   3                 1                1
1829                   1                 2                1
1830                   1                 1                1
1831                   1                 2                1
1832                   2                 2                2
1833                   2                 2                3
1834                   2                 3                3
1835                   2                 2                2
1836                   1                 3                1
1837                   2                 2                2


In [14]:
print("TRAIN rows from test rounds:")
print(
    train[train["round"].isin(test["round"].unique())]
    .shape
)

print("\nTEST rows:")
print(test.shape)

TRAIN rows from test rounds:
(240, 75)

TEST rows:
(240, 74)


In [15]:
print("\nTraining rows per test round:")
print(
    train[train["round"].isin(test["round"].unique())]
    .groupby("round")
    .size()
)

print("\nTest rows per round:")
print(
    test.groupby("round")
    .size()
)


Training rows per test round:
round
1828    24
1829    24
1830    24
1831    24
1832    24
1833    24
1834    24
1835    24
1836    24
1837    24
dtype: int64

Test rows per round:
round
1828    24
1829    24
1830    24
1831    24
1832    24
1833    24
1834    24
1835    24
1836    24
1837    24
dtype: int64


In [16]:
original_test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

print("Does original test match sample?")
print(
    original_test["battle_turn"].equals(
        sample["battle_turn"]
    )
)

Does original test match sample?
True


In [17]:
sorted_test = original_test.sort_values(
    ["round", "turn"]
).reset_index(drop=True)

print("Does sorted test match sample?")
print(
    sorted_test["battle_turn"].equals(
        sample["battle_turn"]
    )
)

Does sorted test match sample?
True


In [18]:
# Compare train and test distributions

for col in X.columns:

    if col in test.columns:

        print("\n==============================")
        print(col)
        print("==============================")

        if pd.api.types.is_numeric_dtype(train[col]):

            print("TRAIN:")
            print(train[col].describe()[["mean", "std", "min", "max"]])

            print("TEST:")
            print(test[col].describe()[["mean", "std", "min", "max"]])

        else:

            print("TRAIN unique:", train[col].nunique())
            print("TEST unique:", test[col].nunique())

            print("Train examples:")
            print(train[col].value_counts().head(5))

            print("Test examples:")
            print(test[col].value_counts().head(5))


opponent_pokemon
TRAIN unique: 17
TEST unique: 17
Train examples:
opponent_pokemon
Charizard    13266
Dragonite     8991
Blastoise     8264
Alakazam      7184
Machamp       5841
Name: count, dtype: int64
Test examples:
opponent_pokemon
Charizard    37
Dragonite    37
Blastoise    27
Gengar       22
Alakazam     19
Name: count, dtype: int64

opponent_type
TRAIN unique: 11
TEST unique: 11
Train examples:
opponent_type
Water      15062
Fire       13252
Rock       10835
Dragon      8983
Psychic     7259
Name: count, dtype: int64
Test examples:
opponent_type
Water     50
Fire      36
Dragon    36
Rock      29
Ghost     22
Name: count, dtype: int64

move_used
TRAIN unique: 9
TEST unique: 9
Train examples:
move_used
Thunderbolt     15698
Quick Attack    11748
Electro Ball    10080
Iron Tail        9174
Volt Tackle      7870
Name: count, dtype: int64
Test examples:
move_used
Thunderbolt     46
Quick Attack    39
Volt Tackle     28
Nuzzle          26
Thunder         26
Name: count, dtype: int6

In [19]:
print("TRAIN trainer_focus_score:")
print(train["trainer_focus_score"].describe())

print("\nTEST trainer_focus_score:")
print(test["trainer_focus_score"].describe())

print("\nTRAIN missing:")
print(train["trainer_focus_score"].isna().sum())

print("TEST missing:")
print(test["trainer_focus_score"].isna().sum())

TRAIN trainer_focus_score:
count    80016.000000
mean        46.442899
std         29.151226
min          0.000000
25%         19.700000
50%         41.500000
75%         71.200000
max        100.000000
Name: trainer_focus_score, dtype: float64

TEST trainer_focus_score:
count    240.000000
mean      55.699583
std       22.993817
min       15.400000
25%       37.400000
50%       55.750000
75%       75.125000
max       95.000000
Name: trainer_focus_score, dtype: float64

TRAIN missing:
0
TEST missing:
0


In [20]:
test_rounds = test["round"].unique()

subset = train[
    train["round"].isin(test_rounds)
].copy()

print("TRAIN rows from test rounds:")
print(subset.shape)

print("\nCorrelation in those rounds:")
print(
    subset[
        ["trainer_focus_score", "pikachu_hp"]
    ].corr()
)

TRAIN rows from test rounds:
(240, 75)

Correlation in those rounds:
                     trainer_focus_score  pikachu_hp
trainer_focus_score             1.000000    0.956813
pikachu_hp                      0.956813    1.000000


In [21]:
import numpy as np

coef = np.polyfit(
    subset["trainer_focus_score"],
    subset["pikachu_hp"],
    1
)

print("\nHP ≈ a * trainer_focus_score + b")
print("a =", coef[0])
print("b =", coef[1])


HP ≈ a * trainer_focus_score + b
a = 0.8311152746213137
b = -11.609746860603313


In [22]:
print("\nTRAIN — test-rounds trainer focus:")
print(
    subset["trainer_focus_score"].describe()
)

print("\nTEST trainer focus:")
print(
    test["trainer_focus_score"].describe()
)


TRAIN — test-rounds trainer focus:
count    240.000000
mean      50.395833
std       29.656402
min        1.600000
25%       22.500000
50%       46.600000
75%       75.725000
max      100.000000
Name: trainer_focus_score, dtype: float64

TEST trainer focus:
count    240.000000
mean      55.699583
std       22.993817
min       15.400000
25%       37.400000
50%       55.750000
75%       75.125000
max       95.000000
Name: trainer_focus_score, dtype: float64


In [23]:
print(train[[
    "trainer_focus_score",
    "pikachu_hp"
]].head(20))

    trainer_focus_score  pikachu_hp
0                  98.6          54
1                  27.7          11
2                  39.4          12
3                  21.5           0
4                  66.8          29
5                  35.5          15
6                 100.0          54
7                  81.5          47
8                  59.3          31
9                  97.7          54
10                 95.8          54
11                 38.3          11
12                 12.8           2
13                 15.6           0
14                 19.9           0
15                  5.4           0
16                 11.5           0
17                 12.7           0
18                 34.7          18
19                 31.0          12


In [25]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X_focus = train[["trainer_focus_score"]]
y = train["pikachu_hp"]

model_focus = LinearRegression()

model_focus.fit(X_focus, y)

train_pred = model_focus.predict(X_focus)

print("Training R²:",
      r2_score(y, train_pred))

print("\nCoefficient:",
      model_focus.coef_[0])

print("Intercept:",
      model_focus.intercept_)

Training R²: 0.9254399887660985

Coefficient: 0.7925108181407473
Intercept: -10.818234973057791


In [26]:
test_focus_pred = model_focus.predict(
    test[["trainer_focus_score"]]
)

print("\nTest predictions:")
print(
    pd.Series(test_focus_pred).describe()
)


Test predictions:
count    240.000000
mean      33.324287
std       18.222848
min        1.386432
25%       18.821670
50%       33.364243
75%       48.719140
max       64.470293
dtype: float64


In [27]:
important_cols = [
    "pikachu_level",
    "opponent_level",
    "move_power",
    "attack_stat",
    "defense_stat",
    "sp_attack_stat",
    "sp_defense_stat",
    "speed_stat_pikachu",
    "speed_stat_opponent",
    "type_effectiveness",
    "damage_dealt",
    "healing_applied",
    "previous_hp",
    "max_hp"
]

print(
    train[
        ["trainer_focus_score"] + important_cols
    ].corr()["trainer_focus_score"]
    .sort_values(ascending=False)
)

trainer_focus_score    1.000000
previous_hp            0.396223
healing_applied        0.102224
defense_stat           0.033954
sp_defense_stat        0.030970
opponent_level         0.019540
speed_stat_opponent   -0.004374
speed_stat_pikachu    -0.005299
pikachu_level         -0.007514
max_hp                -0.010732
sp_attack_stat        -0.011511
attack_stat           -0.016785
type_effectiveness    -0.173372
move_power            -0.309223
damage_dealt          -0.341046
Name: trainer_focus_score, dtype: float64


In [31]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

features_test = [
    "trainer_focus_score",
    "previous_hp",
    "damage_dealt",
    "healing_applied",
    "move_power",
    "type_effectiveness",
    "move_hit",
    "max_hp"
]

X = train[features_test]
y = train["pikachu_hp"]

# Fill missing numerical values
imputer = SimpleImputer(strategy="median")

X_clean = imputer.fit_transform(X)

# Train
model_test = LinearRegression()
model_test.fit(X_clean, y)

# Predict

pred = model_test.predict(X_clean)

print("Training R²:", r2_score(y, pred))

Training R²: 0.9467991341975057


In [33]:
test_rounds = test["round"].unique()

train_test_rounds = train[
    train["round"].isin(test_rounds)
].copy()

X_round = train_test_rounds[features_test]
y_round = train_test_rounds["pikachu_hp"]

# IMPORTANT: use the SAME imputer fitted on training data
X_round_clean = imputer.transform(X_round)

pred_round = model_test.predict(X_round_clean)

print(
    "Test-round R²:",
    r2_score(y_round, pred_round)
)

Test-round R²: 0.9438760075288963


In [34]:
compare_cols = [
    "round",
    "turn",
    "trainer_focus_score",
    "previous_hp",
    "damage_dealt",
    "healing_applied",
    "move_power",
    "type_effectiveness",
    "move_hit",
    "max_hp"
]

print("TRAIN:")
print(train[compare_cols].tail(10))

print("\nTEST:")
print(test[compare_cols].head(10))

TRAIN:
       round  turn  trainer_focus_score  previous_hp  damage_dealt  \
80006   3334    14                 81.2          NaN          30.0   
80007   3334    15                 59.5         44.0          10.0   
80008   3334    16                 23.0         32.0          20.0   
80009   3334    17                 59.6         13.0          35.0   
80010   3334    18                 53.8         76.0          55.0   
80011   3334    19                 39.1         19.0          40.0   
80012   3334    20                 91.8         76.0           0.0   
80013   3334    21                 54.9         76.0         140.0   
80014   3334    22                 91.5         76.0          30.0   
80015   3334    23                 53.8         48.0          35.0   

       healing_applied  move_power  type_effectiveness  move_hit  max_hp  
80006              0.0        90.0                 0.5       1.0    76.0  
80007              0.0        80.0                 0.5       1.0    76.0

In [35]:
print("\nTRAIN rows:", len(train))
print("TEST rows:", len(test))

print("\nTrain round range:")
print(train["round"].min(), train["round"].max())

print("\nTest round range:")
print(test["round"].min(), test["round"].max())

print("\nTrain turn counts:")
print(train["turn"].value_counts().sort_index())

print("\nTest turn counts:")
print(test["turn"].value_counts().sort_index())


TRAIN rows: 80016
TEST rows: 240

Train round range:
1 3334

Test round range:
1828 1837

Train turn counts:
turn
0     3334
1     3334
2     3334
3     3334
4     3334
5     3334
6     3334
7     3334
8     3334
9     3334
10    3334
11    3334
12    3334
13    3334
14    3334
15    3334
16    3334
17    3334
18    3334
19    3334
20    3334
21    3334
22    3334
23    3334
Name: count, dtype: int64

Test turn counts:
turn
0     10
1     10
2     10
3     10
4     10
5     10
6     10
7     10
8     10
9     10
10    10
11    10
12    10
13    10
14    10
15    10
16    10
17    10
18    10
19    10
20    10
21    10
22    10
23    10
Name: count, dtype: int64


In [36]:
compare_cols = [
    "round",
    "turn",
    "trainer_focus_score",
    "previous_hp",
    "damage_dealt",
    "healing_applied",
    "move_power",
    "type_effectiveness",
    "move_hit",
    "max_hp"
]

train_lookup = train.set_index(["round", "turn"])
test_lookup = test.set_index(["round", "turn"])

common_index = test_lookup.index.intersection(train_lookup.index)

comparison = pd.DataFrame({
    "train_trainer_focus": train_lookup.loc[common_index, "trainer_focus_score"].values,
    "test_trainer_focus": test_lookup.loc[common_index, "trainer_focus_score"].values,

    "train_previous_hp": train_lookup.loc[common_index, "previous_hp"].values,
    "test_previous_hp": test_lookup.loc[common_index, "previous_hp"].values,

    "train_damage": train_lookup.loc[common_index, "damage_dealt"].values,
    "test_damage": test_lookup.loc[common_index, "damage_dealt"].values,

    "train_move_power": train_lookup.loc[common_index, "move_power"].values,
    "test_move_power": test_lookup.loc[common_index, "move_power"].values,

    "train_effectiveness": train_lookup.loc[common_index, "type_effectiveness"].values,
    "test_effectiveness": test_lookup.loc[common_index, "type_effectiveness"].values
})

print(comparison.head(20))

    train_trainer_focus  test_trainer_focus  train_previous_hp  \
0                  36.1                64.4               10.0   
1                  95.3                70.4               81.0   
2                   8.6                40.6               81.0   
3                  79.0                34.3               81.0   
4                  54.5                25.8               44.0   
5                  68.4                71.9               21.0   
6                  60.3                90.4                4.0   
7                 100.0                31.9               81.0   
8                  82.2                41.1               59.0   
9                  40.3                64.2               20.0   
10                 92.9                88.5               81.0   
11                 35.6                68.4               81.0   
12                 88.0                17.3               81.0   
13                 44.9                77.2               81.0   
14        

In [37]:
print(
    "\nIdentical rows:",
    (
        comparison["train_trainer_focus"]
        == comparison["test_trainer_focus"]
    ).sum()
)

print(
    "Total rows:",
    len(comparison)
)


Identical rows: 0
Total rows: 240


In [38]:
check_cols = [
    "trainer_focus_score",
    "previous_hp",
    "damage_dealt",
    "healing_applied",
    "move_power",
    "type_effectiveness",
    "move_hit",
    "max_hp"
]

for col in check_cols:
    print(f"\n===== {col} =====")
    print("TRAIN")
    print(train[col].describe())

    print("TEST")
    print(test[col].describe())


===== trainer_focus_score =====
TRAIN
count    80016.000000
mean        46.442899
std         29.151226
min          0.000000
25%         19.700000
50%         41.500000
75%         71.200000
max        100.000000
Name: trainer_focus_score, dtype: float64
TEST
count    240.000000
mean      55.699583
std       22.993817
min       15.400000
25%       37.400000
50%       55.750000
75%       75.125000
max       95.000000
Name: trainer_focus_score, dtype: float64

===== previous_hp =====
TRAIN
count    77737.000000
mean        46.220873
std         22.887874
min          1.000000
25%         27.000000
50%         52.000000
75%         65.000000
max         81.000000
Name: previous_hp, dtype: float64
TEST
count    229.000000
mean      53.711790
std       26.239867
min        1.000000
25%       30.000000
50%       58.000000
75%       81.000000
max       81.000000
Name: previous_hp, dtype: float64

===== damage_dealt =====
TRAIN
count    77571.000000
mean        40.137294
std         46.41747

In [39]:
for col in check_cols:
    train_mean = train[col].mean()
    test_mean = test[col].mean()

    print(
        f"{col:25s} "
        f"Train={train_mean:.2f} "
        f"Test={test_mean:.2f} "
        f"Difference={test_mean-train_mean:.2f}"
    )

trainer_focus_score       Train=46.44 Test=55.70 Difference=9.26
previous_hp               Train=46.22 Test=53.71 Difference=7.49
damage_dealt              Train=40.14 Test=44.08 Difference=3.94
healing_applied           Train=1.40 Test=2.41 Difference=1.01
move_power                Train=71.61 Test=70.82 Difference=-0.79
type_effectiveness        Train=1.03 Test=1.02 Difference=-0.01
move_hit                  Train=0.94 Test=0.94 Difference=0.00
max_hp                    Train=66.11 Test=81.00 Difference=14.89
